In [2]:
import importlib
import script
import NHL_script

# Reload the script after making changes
importlib.reload(script)

# RUNNING THE SCRIPT 
# yesterdays_report_text.txt
# schedule_data.json
# schedule_text.txt
# standings_text.txt
# ERA_leader_data.json
# SO9_leader_data.json
# HR_leader_data.json
# teams_playing_today_data.json
# team_data.json
# ballpark_data.json
# pitcher_data.json
# batter_data.json
# yesterday_home_run_data.json

# DATES
print(' -----=======-----')
print('script getting date')
date = script.get_date()


# YESTERDAYS REPORT
print(' -----=======-----')
print('script getting yesterdays report')
yesterdays_report = script.get_yesterdays_report()
script.save_list_to_text(yesterdays_report,'yesterdays_report_text')


# SCHEDULE INFO
print(' -----=======-----')
print('script getting schedule')
schedule = script.get_schedule_by_date(date)
processed_schedule = script.process_the_schedule(schedule)
raw_schedule_text = script.get_schedule_text()
schedule_text = script.process_schedule_text(raw_schedule_text)
script.save_to_json(schedule,'schedule_data')
script.save_to_text(schedule_text, 'schedule_text')


# STANDINGS
print(' -----=======-----')
print('script getting schedule')
standings_text = script.get_standings_text()
script.save_to_text(standings_text, "standings_text")


# LEAGUE LEADERS
print(' -----=======-----')
print('script getting league leaders')
eras_leaders = script.league_leaders_era()
so9_leaders = script.league_leaders_strikeouts_per_9_innings()
hr_leaders = script.league_leaders_hrs()
script.save_to_json(eras_leaders, 'ERA_leader_data')
script.save_to_json(so9_leaders, 'SO9_leader_data')
script.save_to_json(hr_leaders, 'HR_leader_data')


# TEAMS PLAYING TODAY and History and Records
print(' -----=======-----')
print('script getting team info')
teams_today = script.get_teams_playing_today_from_processed_schedule(processed_schedule)
team_history = script.get_team_history(teams_today)
team_wins = script.get_team_records(team_history)
ballparks = script.scrape_ballparks_table_to_json()
script.save_to_json(teams_today, 'teams_playing_today_data')
script.save_to_json(team_wins, 'team_data')
script.save_to_json(ballparks, 'ballpark_data')


# PITCHERS
print(' -----=======-----')
print('script  getting pitchers')
pitchers_today = script.process_pitchers_from_processed_schedule(processed_schedule)
processed_pitchers = script.add_stats_to_pitchers(pitchers_today)
script.save_to_json(processed_pitchers,"pitcher_data")


# BATTERS
print(' -----=======-----')
print('script getting batters and yesterdays homers')
rooster = script.process_players_from_roster_into_list(processed_schedule)
batters = script.add_stats_to_batters(rooster)
script.save_to_json(batters,"batter_data")
yesterdays_home_runs = script.get_yesterdays_homers()
script.save_to_json(yesterdays_home_runs,'yesterday_home_run_data')


# CONVERTS
# json data
# 'data/yesterdays_report_text.txt'
# 'data/schedule_text.txt'
# 'data/standings_text.txt'
# 'data/schedule_data.json'
schedule_data = 'data/schedule_data.json'
script.save_to_html(script.json_to_html_table(schedule_data), "schedule_data")
era_leader = 'data/ERA_leader_data.json'
script.save_to_html(script.json_to_html_table(era_leader), "era_leader_text")
s09_leader = 'data/SO9_leader_data.json'
script.save_to_html(script.json_to_html_table(s09_leader), "s09_leader_text")
hr_leader = 'data/HR_leader_data.json'
script.save_to_html(script.json_to_html_table(hr_leader), "hr_leader_text")
teams_today = 'data/teams_playing_today_data.json'
script.save_to_html(script.json_to_html_table(teams_today), "teams_today_text")
team_data = 'data/team_data.json'
script.save_to_html(script.json_to_html_table(team_data), "team_data_text")
ballpark_data = 'data/ballpark_data.json'
script.save_to_html(script.json_to_html_table(ballpark_data), "ballpark_data_text")
pitcher_data = 'data/pitcher_data.json'
script.save_to_html(script.json_to_html_table(pitcher_data), "pitcher_data_text")
batter_data = 'data/batter_data.json'
script.save_to_html(script.json_to_html_table(batter_data), "batter_data_text")
yesterday_homerun_data = 'data/yesterday_home_run_data.json'
script.save_to_html(script.json_to_html_table(yesterday_homerun_data), "yesterday_homerun_data_text")

import importlib
import script
import NHL_script
import mlbstatsapi
import statsapi

# Reload the script after making changes
importlib.reload(script)

def get_season_stats(p_id):
    try:
        beans = statsapi.player_stat_data(p_id, group="hitting", type="season")
    except:
        beans = {}
    # beans = statsapi.player_stat_data(p_id, group="[hitting]", type="season", sportId=1, season=2026)
    return beans

batter_data = 'data/batter_data.json'

batters = script.read_json_file(batter_data)
for x in batters:
    # print(x)

    stats_for_player = get_season_stats(x['player_id'])

    # print(stats_for_player)
    try:
        for key,value in stats_for_player.items():
            x.update({key:value})
    except:
        continue


def flatten_dict(
    data,
    parent_key="",
    sep=".",
    flatten_lists=True,
    list_index=0
):
    """
    Flattens a nested dictionary into a single-level dictionary.
    Parameters
    ----------
    data : dict
        Input dictionary
    parent_key : str
        Used internally for recursion
    sep : str
        Separator for nested keys
    flatten_lists : bool
        If True, lists are flattened by taking one element
    list_index : int
        Which index of a list to flatten (default = 0)

    Returns
    -------
    dict
        Flattened dictionary
    """

    items = {}

    for key, value in data.items():
        new_key = f"{parent_key}{sep}{key}" if parent_key else key

        # Case 1: Nested dictionary
        if isinstance(value, dict):
            items.update(flatten_dict(value, new_key, sep, flatten_lists, list_index))

        # Case 2: List (e.g. your "stats")
        elif isinstance(value, list) and flatten_lists:
            if value and isinstance(value[list_index], dict):
                items.update(
                    flatten_dict(
                        value[list_index],
                        new_key,
                        sep,
                        flatten_lists,
                        list_index
                    )
                )
            else:
                items[new_key] = value

        # Case 3: Base value
        else:
            items[new_key] = value

    return items


import unicodedata
import re

def name_to_baseball_reference_anchor(name: str) -> str:
    """
    Convert a player's name into a Baseball-Reference HTML <a> tag.
    """

    # Normalize accents (Ramón → Ramon)
    normalized = unicodedata.normalize("NFKD", name)
    normalized = normalized.encode("ASCII", "ignore").decode("ASCII")

    parts = normalized.strip().split()
    if len(parts) < 2:
        raise ValueError("Name must include at least first and last name")

    first_name = re.sub(r"[^a-z]", "", parts[0].lower())
    last_name = re.sub(r"[^a-z]", "", parts[-1].lower())

    last_initial = last_name[0]
    slug = f"{last_name[:5]}{first_name[:2]}01"

    url = f"https://www.baseball-reference.com/players/{last_initial}/{slug}.shtml"

    return f'<a href="{url}" target="_blank">{name}</a>'


import html

def fix_escaped_links_in_file(file_name: str) -> None:
    """
    Reads an HTML file line by line.
    If a line contains an escaped <a> tag wrapped in <td>...</td>,
    it unescapes the <a> tag and removes the <td> wrapper.
    Overwrites the original file.
    """

    fixed_lines = []

    with open(file_name, "r", encoding="utf-8") as f:
        for line in f:
            stripped = line.strip()

            # Target only <td> that contains an escaped <a> tag
            if stripped.startswith("<td>") and "&lt;a " in stripped:
                # Remove <td> and </td>
                inner = stripped[len("<td>"):-len("</td>")]

                # Unescape HTML entities
                unescaped = html.unescape(inner)
                unescaped = f"<td>{unescaped}</td>"
                fixed_lines.append(unescaped + "\n")
            else:
                fixed_lines.append(line)

    with open(file_name, "w", encoding="utf-8") as f:
        f.writelines(fixed_lines)


flattened_list = [flatten_dict(entry) for entry in batters]

for x in flattened_list:
    link_name = name_to_baseball_reference_anchor(x['player_name'])
    x.update({'p_name':x['player_name']})
    x.update({'player_name':link_name})


batter_data2 = 'batter_data2'
script.save_to_json(flattened_list,batter_data2)
batter_s = 'data/batter_data2.json'
script.save_to_html(script.json_to_html_table(batter_s), "batter_s")
fix_escaped_links_in_file('data/batter_s.html')


 -----=======-----
script getting date
 -----=======-----
script getting yesterdays report
Archived file already exists: data/archived_data/yesterdays_report_text_2026-04-19.txt
Today's data saved to data/yesterdays_report_text.txt
 -----=======-----
script getting schedule
Archived file already exists: data/archived_data/schedule_data_2026-04-19.json
Today's data saved to data/schedule_data.json
Archived file already exists: data/archived_data/schedule_text_2026-04-19.txt
Today's data saved to data/schedule_text.txt
 -----=======-----
script getting schedule
Archived file already exists: data/archived_data/standings_text_2026-04-19.txt
Today's data saved to data/standings_text.txt
 -----=======-----
script getting league leaders
Archived file already exists: data/archived_data/ERA_leader_data_2026-04-19.json
Today's data saved to data/ERA_leader_data.json
Archived file already exists: data/archived_data/SO9_leader_data_2026-04-19.json
Today's data saved to data/SO9_leader_data.json
Ar

In [ ]:


# BATTERS
# print('script getting batters and yesterdays homers')
# rooster = script.process_players_from_roster_into_list(processed_schedule)
# batters = script.add_stats_to_batters(rooster)
# script.save_to_json(processed_batters,"batter_data")
# batters_with_streaks = script.process_batters(batters,team_history)
# batter_vs_pitcher = script.old_batter_vs_pitchers_get()
# batter_vs_pitcher_with_streaks = script.get_streaks_for_bvp(batter_vs_pitcher,batters_with_streaks)
# todays_dh_batters = script.find_dh_batters_add_stats_streaks(schedule, batters_with_streaks)
# yesterdays_home_runs = script.get_yesterdays_homers()
# script.save_to_json(batters_with_streaks, "batter_data")
# script.save_to_json(batter_vs_pitcher_with_streaks,'batter_vs_pitcher_data')
# script.save_to_json(todays_dh_batters, 'dh_batter_data')
# script.save_to_json(yesterdays_home_runs,'yesterday_home_run_data')

# # analysis of home runs
# bvp_data = script.read_json_list('data/batter_vs_pitcher_data.json')

# for a in bvp_data:
#     hr = a.get('all_HR_record', '')
#     hr_analysis_dict = script.analyze_score_sequence(hr)
#     a.update({'all_HR_analysis': hr_analysis_dict})
#     h = a.get('all_H_record', '')
#     h_analysis_dict = script.analyze_score_sequence(h)
#     a.update({'all_H_analysis': h_analysis_dict})
#     rbi = a.get('all_RBI_record', '')
#     rbi_analysis_dict = script.analyze_score_sequence(rbi)
#     a.update({'all_RBI_analysis': rbi_analysis_dict})

# script.save_to_json(bvp_data, "batter_vs_pitcher_data")


# print('make index')
# index_html = script.make_index()
# script.save_to_text(index_html, 'raw_index')

# index_html = make_index()
# processed_html = script.process_html(index_html)
# print(processed_html)
# script.save_to_html(processed_html,'index')

# NHL_script.process_schedule()
# NHL_script.process_yesterdays_scores_to_report()
# NHL_script.process_raw_skaters_html_table()


HTTPError: 400 Client Error: Bad Request for url: https://statsapi.mlb.com/api/v1/people/?hydrate=stats(group=%5Bhitting%5D,type=season,sportId=1),currentTeam

In [5]:
for x in batters:
    print(x)

{'player_name': 'Brad Lord', 'team': 'Washington Nationals', 'team_id': 120, 'player_id': 695418, 'id': 695418, 'first_name': 'Brad', 'last_name': 'Lord', 'active': True, 'current_team': 'Washington Nationals', 'position': 'P', 'nickname': None, 'last_played': None, 'mlb_debut': '2025-03-30', 'bat_side': 'Right', 'pitch_hand': 'Right', 'stats': []}
{'player_name': 'Brady House', 'team': 'Washington Nationals', 'team_id': 120, 'player_id': 691781, 'position': '3B', 'games_played': 17.0, 'H': 16.0, 'HR': 2.0, 'RBI': 7.0, 'HRpg': 0.118, 'fHRpg': '1/7', 'Hpg': 0.941, 'fHpg': '1', 'RBIpg': 0.412, 'fRBIpg': '2/5', 'HR24': 4, 'HR24pg': 0.05, 'fHR24pg': '0', 'id': 691781, 'first_name': 'Brady', 'last_name': 'House', 'active': True, 'current_team': 'Washington Nationals', 'nickname': None, 'last_played': None, 'mlb_debut': '2025-06-16', 'bat_side': 'Right', 'pitch_hand': 'Right', 'stats': [{'type': 'season', 'group': 'hitting', 'season': '2026', 'stats': {'age': 23, 'gamesPlayed': 17, 'groundOu